# Dataset Loading
Load your self-generated digits dataset from `dataset/labels.csv` and `dataset/images/*`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()  # coursework.ipynb is in /project
LABELS_CSV = PROJECT_DIR / 'dataset' / 'labels.csv'

df = pd.read_csv(LABELS_CSV)
df.head()

cur dir :  /Users/raheemkhan/Documents/3rd-Year/Principles_of_AI/project
labels address :  /Users/raheemkhan/Documents/3rd-Year/Principles_of_AI/project/dataset/labels.csv
Rows in labels.csv: 1200


,filepath,label,writer_id,sheet_id,source_sheet,cell_index
0,dataset/images/0/ Khan_ 0_ sheet1_ 0.png,0,Khan,sheet1,Khan_0_sheet1.JPG,0
1,dataset/images/0/ Khan_ 0_ sheet1_ 1.png,0,Khan,sheet1,Khan_0_sheet1.JPG,1
2,dataset/images/0/ Khan_ 0_ sheet1_ 2.png,0,Khan,sheet1,Khan_0_sheet1.JPG,2
3,dataset/images/0/ Khan_ 0_ sheet1_ 3.png,0,Khan,sheet1,Khan_0_sheet1.JPG,3
4,dataset/images/0/ Khan_ 0_ sheet1_ 4.png,0,Khan,sheet1,Khan_0_sheet1.JPG,4


In [6]:
# Loading the dataset
X_list ,y_list = [],[]
for row in df.itertuples(index=False):
    img_path = row.filepath
    img = Image.open(img_path).convert("L").resize((28,28))
    X_list.append(np.array(img,dtype=np.uint8))
    y_list.append(int(row.label))

X = np.stack(X_list)
y = np.array(y_list, dtype=np.int64)

print('x : ', X)
print("y : ", y)

x :  [[[ 76 158 115 ... 172 155 140]
  [119 239 251 ... 251 249 206]
  [116 237 255 ... 255 255 221]
  ...
  [135 237 255 ... 255 255 255]
  [118 237 255 ... 255 255 255]
  [112 234 255 ... 255 255 255]]

 [[123 132 120 ... 181 150 146]
  [255 255 255 ... 255 255 255]
  [255 255 255 ... 255 255 255]
  ...
  [255 255 255 ... 255 255 255]
  [255 255 255 ... 255 255 255]
  [255 255 255 ... 255 255 255]]

 [[105 110 133 ... 187 210 212]
  [113 210 251 ... 255 255 255]
  [115 221 255 ... 255 255 255]
  ...
  [149 246 255 ... 255 255 255]
  [174 255 255 ... 255 255 255]
  [171 255 255 ... 255 255 255]]

 ...

 [[250 250 250 ... 250 250 250]
  [250 250 250 ... 250 250 250]
  [250 250 250 ... 250 250 250]
  ...
  [250 250 250 ... 250 250 250]
  [250 250 250 ... 250 250 250]
  [250 223 206 ... 229 213 212]]

 [[250 250 250 ... 250 250 161]
  [250 250 250 ... 250 250 199]
  [250 250 250 ... 250 250 191]
  ...
  [250 250 250 ... 250 250 241]
  [250 251 245 ... 250 250 250]
  [206 206 198 ... 238 

In [8]:
# Normalising the pixel values
X_float = X.astype(np.float32) / 255.0
X_flat = X_float.reshape(len(X_float), -1)  # (N, 784)

print(X_float.dtype, X_float)



float32 [[[0.29803923 0.61960787 0.4509804  ... 0.6745098  0.60784316 0.54901963]
  [0.46666667 0.9372549  0.9843137  ... 0.9843137  0.9764706  0.80784315]
  [0.45490196 0.92941177 1.         ... 1.         1.         0.8666667 ]
  ...
  [0.5294118  0.92941177 1.         ... 1.         1.         1.        ]
  [0.4627451  0.92941177 1.         ... 1.         1.         1.        ]
  [0.4392157  0.91764706 1.         ... 1.         1.         1.        ]]

 [[0.48235294 0.5176471  0.47058824 ... 0.70980394 0.5882353  0.57254905]
  [1.         1.         1.         ... 1.         1.         1.        ]
  [1.         1.         1.         ... 1.         1.         1.        ]
  ...
  [1.         1.         1.         ... 1.         1.         1.        ]
  [1.         1.         1.         ... 1.         1.         1.        ]
  [1.         1.         1.         ... 1.         1.         1.        ]]

 [[0.4117647  0.43137255 0.52156866 ... 0.73333335 0.8235294  0.83137256]
  [0.44313726 

In [10]:
# using sklean to split the train dataset
from sklearn.model_selection import train_test_split
import random

SEED = 42 # taking the usual number

random.seed(SEED)
np.random.seed(SEED)

X_train, X_test, y_train, y_test = train_test_split(
    X_flat, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
    shuffle=True
)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)


(960, 784) (240, 784) (960,) (240,)


In [14]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(
    # multi_class="multinomial",  getting warning that multiclass is deprecated and thbe default is multinomial so going to remove this for now
    solver="lbfgs",
    max_iter=2000,
    random_state=42
)
clf.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


In [16]:
y_pred = clf.predict(X_test)
probs = clf.predict_proba(X_test) # shape: (num_samples, num_classes)

In [17]:
from sklearn.metrics import accuracy_score, classification_report, log_loss
print("Accuracy : ", accuracy_score(y_test, y_pred))
print("Log loss : ", log_loss(y_test, probs))
print(classification_report(y_test, y_pred))


Accuracy :  0.6916666666666667
Log loss :  0.8749494520132209
              precision    recall  f1-score   support

           0       0.56      0.50      0.53        40
           1       0.80      0.88      0.83        40
           2       1.00      0.95      0.97        40
           3       0.62      0.70      0.66        40
           4       0.49      0.50      0.49        40
           5       0.69      0.62      0.66        40

    accuracy                           0.69       240
   macro avg       0.69      0.69      0.69       240
weighted avg       0.69      0.69      0.69       240

